# 06 — Visualize Predictions: CatBoost vs the Best Temporal Model

Loads the predictions saved by `predict.py` — CatBoost, the production model that actually answers the thesis question (see its module docstring in `prod/predict.py`) — together with the historical `df_development.parquet`, and compares them against the best model from `05_temporal_persistence_check.ipynb` (SARIMAX + 1 exogenous feature, `Alcohol use disorders`), forecast forward to the same 2022-2023 period.

**Why compare against a model that notebook already showed answers a different question.** The point isn't to pick a "better" forecast — it's to see, country by country, whether a model built purely from socioeconomic/mental-health determinants (CatBoost) and one built purely from a country's own persistence plus one determinant (SARIMAX + exog) end up telling a similar story or diverging. Agreement is a mild corroborating signal; disagreement isn't a bug in either model, just a reminder that they were built to answer different questions and aren't expected to coincide exactly.

**Where the temporal forecast comes from.** SARIMAX's `.forecast()` walks forward step by step from the end of its training data — to reach 2022-2023 from a model fit through ~2014, every intervening year (the original test/val years) has to be forecast too, using the real historical values of the exogenous feature for those years, and only the final two years are kept. This is computed inline here, not loaded from a persisted file — it's a comparison plot for this notebook, not a second production inference path alongside `predict.py`.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src import (
    TARGET,
    temporal_split,
    plot_predictions_trend,
    plot_predictions_by_country,
    plot_predictions_model_comparison,
    plot_predictions_by_country_comparison,
    save_figure,
    save_artifact,
)
from src.timeseries_models import fit_sarimax_models, forecast_sarimax

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)

fig_prefix = "06_"

MODEL_A_NAME = "CatBoost"
MODEL_B_NAME = "SARIMAX +1 exog"
TEMPORAL_EXOG_FEATURE = "Alcohol use disorders"  # matches 05_temporal_persistence_check.ipynb's choice
TEMPORAL_ORDER = (1, 1, 1)  # (1,1,0) was the original default; 05b_temporal_persistence_improvements.ipynb found (1,1,1) beats it on both Test and Val R2

SPOTLIGHT_COUNTRIES = [("LTU", "Lithuania"), ("GRC", "Greece"), ("DEU", "Germany")]

df_history = pd.read_parquet("../data/processed/df_development.parquet")
df_real_world = pd.read_parquet("../data/processed/df_real_world.parquet")
df_predictions_catboost = pd.read_parquet("../outputs/tables/predictions.parquet")
print(f"CatBoost predictions: {len(df_predictions_catboost)} rows, years: {sorted(df_predictions_catboost['Year'].unique().tolist())}")

## Forecasting the temporal model forward to 2022-2023

Fits SARIMAX (+`Alcohol use disorders`) per country on Option B's training years only — same as `05_temporal_persistence_check.ipynb` — then extends the forecast through the test and validation years to reach 2022-2023.

In [ ]:
df_train_B, df_test_B, df_val_B, *_ = temporal_split(df_history)

sarimax_models = fit_sarimax_models(df_train_B, TARGET, exog_features=[TEMPORAL_EXOG_FEATURE], order=TEMPORAL_ORDER)
print(f"Fitted {len(sarimax_models)}/{df_train_B['Code'].nunique()} country models")

# Persisted mainly for later inspection/reuse without re-fitting — this
# notebook always fits fresh each run rather than loading an existing artifact.
models_path = save_artifact(sarimax_models, "../outputs/models/sarimax_exog_models.joblib")
print(f"SARIMAX (+{TEMPORAL_EXOG_FEATURE}) models saved: {models_path}")

future_block = pd.concat([df_test_B, df_val_B, df_real_world]).sort_values(["Code", "Year"])

rows = []
for code, group in df_real_world.groupby("Code"):
    if code not in sarimax_models:
        continue
    country_future = future_block[future_block["Code"] == code]
    forecast = forecast_sarimax(sarimax_models, code, country_future, exog_features=[TEMPORAL_EXOG_FEATURE])
    for _, row in group.iterrows():
        rows.append({
            "Country": row["Country"], "Code": code, "Year": row["Year"],
            "Predicted suicide rate": forecast.loc[row["Year"]],
        })

df_predictions_temporal = pd.DataFrame(rows)
df_predictions_temporal.to_parquet("../outputs/tables/predictions_temporal.parquet", index=False)
print(f"Saved: outputs/tables/predictions_temporal.parquet ({len(df_predictions_temporal)} rows)")
display(df_predictions_temporal.head())

## CatBoost predictions alone

Same spotlight-country trend plots and by-year bar charts as before — the production model's view on its own, before comparing it to anything.

In [ ]:
for code, name in SPOTLIGHT_COUNTRIES:
    fig = plot_predictions_trend(df_history, df_predictions_catboost, code, name)
    save_figure(fig, name=f"predictions_trend_{code.lower()}", prefix=fig_prefix)

In [ ]:
for year in sorted(df_predictions_catboost["Year"].unique()):
    fig = plot_predictions_by_country(df_predictions_catboost, year)
    save_figure(fig, name=f"predictions_by_country_{year}", prefix=fig_prefix)

## CatBoost vs SARIMAX +1 exog, side by side

The comparison the rest of this notebook exists for — both models' forecasts for the same countries and years, overlaid on the actual history.

In [ ]:
for code, name in SPOTLIGHT_COUNTRIES:
    fig = plot_predictions_model_comparison(
        df_history, df_predictions_catboost, df_predictions_temporal,
        MODEL_A_NAME, MODEL_B_NAME, code, name,
    )
    save_figure(fig, name=f"predictions_comparison_trend_{code.lower()}", prefix=fig_prefix)

In [ ]:
for year in sorted(df_predictions_catboost["Year"].unique()):
    fig = plot_predictions_by_country_comparison(
        df_predictions_catboost, df_predictions_temporal, MODEL_A_NAME, MODEL_B_NAME, year,
    )
    save_figure(fig, name=f"predictions_comparison_by_country_{year}", prefix=fig_prefix)

## Aggregated to the a priori EU regions

The same comparison, averaged up to the `EU_REGIONS` grouping used descriptively in `02_eda.ipynb` and `04_clustering.ipynb` — countries within a region are averaged for both the trend and bar-chart views, so it's visible whether each model's predictions line up with that regional grouping once aggregated, not just at the individual-country level above.

In [ ]:
from src import EU_REGIONS, plot_predictions_by_region_comparison, plot_predictions_trend_by_region

df_history["Region"] = df_history["Code"].map(EU_REGIONS)
df_predictions_catboost["Region"] = df_predictions_catboost["Code"].map(EU_REGIONS)
df_predictions_temporal["Region"] = df_predictions_temporal["Code"].map(EU_REGIONS)

In [ ]:
for region_name in sorted(set(EU_REGIONS.values())):
    fig = plot_predictions_trend_by_region(
        df_history, df_predictions_catboost, df_predictions_temporal,
        MODEL_A_NAME, MODEL_B_NAME, region_name,
    )
    save_figure(fig, name=f"predictions_comparison_trend_region_{region_name.lower().replace('/', '_').replace(' ', '_')}", prefix=fig_prefix)

In [ ]:
for year in sorted(df_predictions_catboost["Year"].unique()):
    fig = plot_predictions_by_region_comparison(
        df_predictions_catboost, df_predictions_temporal, MODEL_A_NAME, MODEL_B_NAME, year,
    )
    save_figure(fig, name=f"predictions_comparison_by_region_{year}", prefix=fig_prefix)

## Reading the comparison

[Suposición — a completar tras inspeccionar las figuras generadas arriba con tus propios datos, no una conclusión genérica:] compara, país a país, si ambos modelos coinciden en la dirección (sube/baja) aunque difieran en el nivel exacto — coincidencia en la dirección es una señal de robustez cualitativa; discrepancias grandes en países concretos son candidatas a discutir individualmente en la memoria, no un fallo a "arreglar" eligiendo el modelo que más te convenga.